# Γ1 headline gate — context-residual consolidation, n=10 wikitext-2

Per the precommit at [notes/notes/2026-05-27-path-gamma-gamma1-context-residual-precommit.md](https://github.com/) §"Operating point" and the revised graduation criterion at [phase-3-deep-dive.md:188-205](https://github.com/) (CI-disjoint at n ≥ 10 AND per-seed paired robustness ≥ 70%).

**Design.** 2 conditions × 10 seeds = 20 per-seed parallel CUDA subprocesses.

| condition | flags | what it measures |
|---|---|---|
| `gamma1_headline` | `--use-context-residual --no-pull-push --lr-cr 0.1` | The Γ1.c mechanism at the Path C operating point. |
| `pathc_baseline` | (defaults — pull/push on, context-residual off) | Path C reference at matched seeds. Per-seed paired comparison against Γ1 uses these. |

**Pre-flight.** This notebook assumes the Γ1 implementation has been pushed to remote on branch `codex/phase5-prime-bundle-first-scene-memory`. The verify cell (cell 1) fails fast if the flag `use_context_residual` is not present in `src/energy_memory/phase34/online_codebook.py`. If verify fails, push the local branch and re-run.

**Headline criterion (revised 2026-05-27).** Both clauses must hold for Γ1 to graduate Phase 3:

1. Wilson CI on the standard-vs-shuffled-token-control Δ R@K is strictly disjoint in at least one regime stratum at n=10 (`default/spread` or `calibrated/tight`).
2. Per-seed paired robustness ≥ 70%: at least 7/10 seeds show stratum-pooled per-seed Δ > 0.

The aggregation cell at the bottom prints PASS/PARTIAL/FAIL per the revised criterion and emits a per-seed paired comparison against `pathc_baseline`.

In [ ]:
# 1. Clone the repo + verify Γ1 implementation is present on the branch.
#    Fails fast if `use_context_residual` is not found — meaning the local
#    branch was not pushed before launching this notebook.
import subprocess, sys, os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)
# Verify Γ1 code lives at the expected path.
online_cb = Path(REPO_DIR) / 'src' / 'energy_memory' / 'phase34' / 'online_codebook.py'
src = online_cb.read_text()
if 'use_context_residual' not in src:
    raise SystemExit(
        'use_context_residual flag not found in online_codebook.py — '
        'push the local branch first, then re-run this notebook.'
    )
if '_apply_context_residual' not in src:
    raise SystemExit(
        '_apply_context_residual method not found — Γ1 implementation '
        'not present on the remote branch. Push and re-run.'
    )
# Verify CLI flag is present in driver.
driver_src = (Path(REPO_DIR) / 'experiments' / 'c3_phase3_exit_criterion.py').read_text()
if '--use-context-residual' not in driver_src:
    raise SystemExit(
        '--use-context-residual CLI flag not in driver. Push and re-run.'
    )
print('Γ1 implementation verified on branch.')
print('  online_codebook.py: use_context_residual + _apply_context_residual present.')
print('  c3_phase3_exit_criterion.py: --use-context-residual flag present.')
print(f'  current HEAD: {subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()}')


In [ ]:
# 2. Mount Drive for result persistence.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# 3. Install deps. wikitext loader uses HF datasets.
!pip install -q torch datasets huggingface_hub

In [ ]:
# 4. Pre-warm the WikiText-2 HF cache so 20 subprocesses share the on-disk cache.
#    Parent CPU-only (does not init CUDA). Critical: workers must inherit
#    a CUDA-untouched parent.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from datasets import load_dataset
_ = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:1%]')
print('WikiText-2 cache pre-warmed.')

In [ ]:
# 5. GPU info — confirm CUDA available, parent has NOT touched CUDA yet.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv
# Verify parent has not initialized CUDA. The list of compute apps should
# be empty before parallel launch.

In [ ]:
# 6. SMOKE — one tiny Γ1 subprocess on synthetic to confirm runtime is sane.
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/gamma1_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/gamma1_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--use-context-residual', '--no-pull-push', '--lr-cr', '0.1',
       '--D', '128', '--landscape-size', '8',
       '--n-consolidation-events', '50',
       '--vocab-size', '50', '--K', '3', '--beta', '10',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--corpus-source', 'synthetic',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json: {(smoke_out / "c3_summary.json").exists()}')
print('\n=== smoke log (last 30 lines) ===')
!tail -30 {smoke_log}
if rc != 0:
    raise SystemExit('Γ1 smoke failed — abort before parallel launch.')
# Verify Γ1 actually fired in the smoke.
import json
with open(smoke_out / 'c3_summary.json') as f:
    smoke_summary = json.load(f)
ctx_residual_seen = smoke_summary.get('header', {}).get('use_context_residual')
if not ctx_residual_seen:
    raise SystemExit(
        'Γ1 smoke ran but use_context_residual=False in summary — '
        'CLI flag did not propagate. Abort.'
    )
print('\nΓ1 smoke OK; use_context_residual=True confirmed in summary JSON.')

In [ ]:
# 7. PARALLEL launch — 2 conditions × 10 seeds = 20 per-seed subprocesses.
#    Each subprocess runs the C.3 driver at the precommit's wikitext
#    operating point. The only differences between conditions are the
#    Γ1 flags (--use-context-residual, --no-pull-push, --lr-cr).
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

# Operating point per the precommit §"Operating point". All other knobs
# (D, β, K, landscape_size, window_size, vocab_cap, α_anti, repulsion_step_size,
# λ_ac, μ_T, τ_T, λ_cc, metastability_obs_rate, drift_ema_rate,
# n_consolidation_events) match Path C values. See
# experiments/c3_phase3_exit_criterion.py:103-128 for the C.2.x defaults.
BASE_FLAGS = [
    '--device', 'cuda',
    '--corpus-source', 'wikitext',
    '--wikitext-name', 'wikitext-2-raw-v1',
    '--vocab-cap', '1000',
    '--window', '8',
    '--D', '4096',
    '--landscape-size', '64',
    '--beta', '10',
    '--K', '5',
    '--n-consolidation-events', '1000',
    '--alpha-anti', '0.01',
    '--repulsion-step-size', '0.05',
    '--lr-pull', '0.1',
    '--lr-push', '0.05',
]

CONDITIONS = [
    # (tag, extra_flags)
    ('gamma1_headline', ['--use-context-residual', '--no-pull-push', '--lr-cr', '0.1']),
    ('pathc_baseline', []),  # defaults: pull/push on, context-residual off
]
SEEDS = list(range(10))

ENTRIES = [(tag, extra, seed) for (tag, extra) in CONDITIONS for seed in SEEDS]
print(f'launching {len(ENTRIES)} per-seed subprocesses '
      f'({len(CONDITIONS)} conditions × {len(SEEDS)} seeds)')

log_root = Path('reports/gamma1_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(tag, seed):
    return f'reports/gamma1_{tag}_seed{seed}_2026-05-27'

def launch(tag, extra_flags, seed):
    out_dir = out_dir_for(tag, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed),
           *BASE_FLAGS,
           *extra_flags,
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_remaining = len(remaining)
    n_done = total - n_remaining
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, '
          f'{n_remaining} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    from collections import Counter
    cond_running = Counter()
    for key in remaining:
        tag = key.rsplit('_seed', 1)[0]
        cond_running[tag] += 1
    for tag, _ in CONDITIONS:
        n_run = cond_running.get(tag, 0)
        n_done_tag = len(SEEDS) - n_run
        print(f'    {tag:>22}: {n_done_tag}/{len(SEEDS)} done')

def kill_all(remaining):
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

# Staggered launch (1.5 s × 20 = 30 s of launch).
procs = {}
for entry in ENTRIES:
    tag, extra, seed = entry
    key = f'{tag}_seed{seed}'
    procs[key] = launch(tag, extra, seed)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>32}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill — Runtime → Interrupt cell 7 first, THEN run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion.py' in line:
        try:
            pid = int(line.strip().split()[0])
            os.kill(pid, signal.SIGKILL)
            killed += 1
        except Exception as e:
            print(f'  failed to kill {line[:60]}: {e}')
print(f'killed {killed} c3 driver processes')
!nvidia-smi --query-compute-apps=pid,process_name --format=csv

In [ ]:
# 8. Copy all per-seed result dirs + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/gamma1_headline_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
TAGS = ['gamma1_headline', 'pathc_baseline']
SEEDS = list(range(10))
count = 0
for tag in TAGS:
    for seed in SEEDS:
        src = f'reports/gamma1_{tag}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/{tag}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/gamma1_logs'):
    shutil.copytree('reports/gamma1_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')
!ls {dst_root} | head -25

In [ ]:
# 9. AGGREGATION — read per-seed JSONs, compute pooled Wilson CIs + per-seed
#    paired robustness, evaluate the revised C.3 criterion.
#
#    Headline criterion (revised 2026-05-27 per phase-3-deep-dive.md:188-205):
#    (1) Wilson CI on Δ R@K strictly disjoint in ≥ 1 regime stratum at n=10.
#    (2) Per-seed paired robustness ≥ 70%: ≥ 7/10 seeds show stratum-pooled
#        per-seed Δ > 0.
#
#    Both clauses must hold for Γ1.c to graduate Phase 3.
import json
from pathlib import Path
from collections import defaultdict

import math

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1.0 + z*z/n
    center = (p + z*z/(2*n)) / denom
    halfwidth = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (max(0.0, center - halfwidth), min(1.0, center + halfwidth))

TAGS = ['gamma1_headline', 'pathc_baseline']
SEEDS = list(range(10))
MODES = ['default', 'calibrated']
PRIMARY_STRATA = ['tight', 'spread']  # the two strata the criterion evaluates

# Load per-seed JSONs.
data = defaultdict(dict)  # data[tag][seed] = parsed JSON
for tag in TAGS:
    for seed in SEEDS:
        p = Path(f'reports/gamma1_{tag}_seed{seed}_2026-05-27/c3_summary.json')
        if p.exists():
            with p.open() as f:
                data[tag][seed] = json.load(f)
        else:
            print(f'MISSING: {p}')

def extract_per_seed_stratum_counts(summary, mode, stratum, condition_key):
    # condition_key in {'standard', 'shuffled_control'}
    for cell in summary.get('per_cell', []):
        if cell.get('theta_prime_mode') != mode:
            continue
        for s in cell.get('strata', []):
            if s.get('stratum') != stratum:
                continue
            cond = s.get(condition_key, {})
            return (cond.get('hits', 0), cond.get('trials', 0))
    return (0, 0)

# Pool across seeds, compute Wilson CIs.
print('=' * 80)
print('CLAUSE 1 — Pooled CI-disjointness (n=10) per stratum / mode')
print('=' * 80)
for tag in TAGS:
    print(f'\n  condition: {tag}')
    for mode in MODES:
        print(f'    mode = {mode}')
        for stratum in PRIMARY_STRATA:
            std_hits = std_trials = ctrl_hits = ctrl_trials = 0
            for seed in SEEDS:
                s = data[tag].get(seed)
                if s is None:
                    continue
                sh, st = extract_per_seed_stratum_counts(s, mode, stratum, 'standard')
                ch, ct = extract_per_seed_stratum_counts(s, mode, stratum, 'shuffled_control')
                std_hits += sh; std_trials += st
                ctrl_hits += ch; ctrl_trials += ct
            if std_trials == 0 or ctrl_trials == 0:
                print(f'      {stratum:>10}: empty stratum')
                continue
            std_p = std_hits / std_trials
            ctrl_p = ctrl_hits / ctrl_trials
            std_ci = wilson_ci(std_hits, std_trials)
            ctrl_ci = wilson_ci(ctrl_hits, ctrl_trials)
            delta = std_p - ctrl_p
            disjoint = std_ci[0] > ctrl_ci[1] or ctrl_ci[0] > std_ci[1]
            mark = '✓' if disjoint else ' '
            print(f'      {stratum:>10}: std={std_p:.3f} {std_ci}  '
                  f'ctrl={ctrl_p:.3f} {ctrl_ci}  Δ={delta:+.3f}  disjoint={mark}')

# Per-seed paired robustness for Γ1 headline.
print('\n' + '=' * 80)
print('CLAUSE 2 — Per-seed paired robustness (gamma1_headline only)')
print('=' * 80)
print('Each seed: stratum-pooled (tight + spread) per-seed Δ = std_p − ctrl_p, default mode')
per_seed_deltas_g1 = {}
per_seed_deltas_pc = {}
for tag, store in [('gamma1_headline', per_seed_deltas_g1),
                   ('pathc_baseline', per_seed_deltas_pc)]:
    for seed in SEEDS:
        s = data[tag].get(seed)
        if s is None:
            continue
        std_h = std_t = ctrl_h = ctrl_t = 0
        for stratum in PRIMARY_STRATA:
            sh, st = extract_per_seed_stratum_counts(s, 'default', stratum, 'standard')
            ch, ct = extract_per_seed_stratum_counts(s, 'default', stratum, 'shuffled_control')
            std_h += sh; std_t += st
            ctrl_h += ch; ctrl_t += ct
        if std_t and ctrl_t:
            store[seed] = std_h/std_t - ctrl_h/ctrl_t

print('\nseed |  Δ_gamma1   Δ_pathc   shift (g1 − pc)   g1 > 0?')
print('-' * 60)
n_g1_pos = 0
n_paired_g1_better = 0
shifts = []
for seed in SEEDS:
    g1 = per_seed_deltas_g1.get(seed)
    pc = per_seed_deltas_pc.get(seed)
    if g1 is None or pc is None:
        print(f' {seed}  |  MISSING')
        continue
    shift = g1 - pc
    shifts.append(shift)
    g1_pos = g1 > 0
    g1_better = g1 > pc
    n_g1_pos += g1_pos
    n_paired_g1_better += g1_better
    marks = '+' if g1_pos else ' '
    print(f' {seed}  |  {g1:+.3f}   {pc:+.3f}     {shift:+.3f}          {marks}')

n_seeds_g1 = sum(1 for s in SEEDS if s in per_seed_deltas_g1)
if n_seeds_g1:
    pct_pos = n_g1_pos / n_seeds_g1 * 100
    g1_mean = sum(per_seed_deltas_g1[s] for s in per_seed_deltas_g1) / n_seeds_g1
    pc_mean = (sum(per_seed_deltas_pc[s] for s in per_seed_deltas_pc) / 
               max(1, len(per_seed_deltas_pc)))
    print(f'\nΓ1 per-seed mean Δ: {g1_mean:+.4f}  (PathC: {pc_mean:+.4f})')
    print(f'Γ1 seeds with Δ > 0: {n_g1_pos}/{n_seeds_g1} ({pct_pos:.0f}%)')
    print(f'Γ1 better than PathC at matched seeds: {n_paired_g1_better}/{n_seeds_g1}')

# Verdict per the revised criterion.
print('\n' + '=' * 80)
print('VERDICT — revised C.3 criterion')
print('=' * 80)
# Clause 1: was any stratum / mode CI-disjoint for gamma1_headline?
clause1 = False
for mode in MODES:
    for stratum in PRIMARY_STRATA:
        std_hits = std_trials = ctrl_hits = ctrl_trials = 0
        for seed in SEEDS:
            s = data['gamma1_headline'].get(seed)
            if s is None: continue
            sh, st = extract_per_seed_stratum_counts(s, mode, stratum, 'standard')
            ch, ct = extract_per_seed_stratum_counts(s, mode, stratum, 'shuffled_control')
            std_hits += sh; std_trials += st; ctrl_hits += ch; ctrl_trials += ct
        if std_trials and ctrl_trials:
            std_ci = wilson_ci(std_hits, std_trials)
            ctrl_ci = wilson_ci(ctrl_hits, ctrl_trials)
            if std_ci[0] > ctrl_ci[1] or ctrl_ci[0] > std_ci[1]:
                clause1 = True
                break
    if clause1: break

# Clause 2: ≥70% of γ1 seeds have positive Δ.
clause2 = (n_seeds_g1 > 0) and (n_g1_pos / n_seeds_g1 >= 0.7)

if clause1 and clause2:
    print('  ✅ PASS — both clauses hold. Γ1.c graduates Phase 3.')
    print('  Next: write Report 113 documenting Γ1.c graduation. Path γ is')
    print('  ready to inform a fresh Phase 5′ reopen precommit.')
elif clause1 and not clause2:
    print(f'  ⚠ PARTIAL — clause 1 holds (CI-disjoint somewhere) but clause 2')
    print(f'    fails ({n_g1_pos}/{n_seeds_g1} = {pct_pos:.0f}% per-seed positive,')
    print(f'    threshold 70%).')
    print(f'  Follow-up F2 (symmetric Γ1.c) or F3 (Γ1.c + Γ3) may push robustness up.')
elif not clause1 and clause2:
    print(f'  ⚠ PARTIAL — clause 2 holds ({pct_pos:.0f}% per-seed positive) but')
    print(f'    clause 1 fails (no stratum CI-disjoint at n=10).')
    print(f'  Consider pooling to n=30 (seeds 10..29) to tighten the CI.')
else:
    print(f'  ❌ FAIL — neither clause holds.')
    print(f'    Clause 1: no stratum CI-disjoint.')
    print(f'    Clause 2: per-seed positive = {n_g1_pos}/{n_seeds_g1} = {pct_pos:.0f}% < 70%.')
    print(f'  Follow-up F1 (lr_cr sweep at smoke scale) is the precommitted next step.')

# Save aggregated JSON to Drive.
import os
agg = {
    'tag': 'gamma1_headline_2026-05-27',
    'precommit': 'notes/notes/2026-05-27-path-gamma-gamma1-context-residual-precommit.md',
    'criterion': 'phase-3-deep-dive.md:188-205 (revised 2026-05-27)',
    'clause1_ci_disjoint': clause1,
    'clause2_per_seed_robust': clause2,
    'gamma1_per_seed_deltas': {str(k): float(v) for k, v in per_seed_deltas_g1.items()},
    'pathc_per_seed_deltas': {str(k): float(v) for k, v in per_seed_deltas_pc.items()},
    'n_gamma1_positive': n_g1_pos,
    'n_gamma1_seeds': n_seeds_g1,
    'pct_gamma1_positive': pct_pos if n_seeds_g1 else 0.0,
}
agg_path = '/content/drive/MyDrive/neuro-ai/results/gamma1_headline_2026-05-27/aggregate.json'
os.makedirs(os.path.dirname(agg_path), exist_ok=True)
with open(agg_path, 'w') as f:
    json.dump(agg, f, indent=2)
print(f'\nwrote aggregate to {agg_path}')